# Controlled RM/PM Data Generation And Lineage

This notebook documents a reproducible data-generating process with known ground truth. It contains 24 finished goods, 120 RM/PM materials, complete BOM mappings, yield, scrap, MOQ, order multiples, lead times, production plans, actual production, shocks and structural changes.

**Claim boundary:** controlled synthetic recovery validates pipeline behavior. It does not prove production accuracy or that all real-data error is caused by data quality.


In [1]:
from pathlib import Path
import json
import pandas as pd
try:
    from IPython.display import Image, display
except ImportError:
    class Image:
        def __init__(self, filename): self.filename = filename
        def __repr__(self): return f"Image({self.filename})"
    def display(value): print(value)

ROOT = Path.cwd()
if ROOT.name != "v8_controlled_synthetic_validation":
    candidate = ROOT / "Ai miroservices/modeling/v8_controlled_synthetic_validation"
    ROOT = candidate if candidate.exists() else ROOT
OUT = ROOT / "outputs"
DATA = OUT / "data"
PLOTS = OUT / "plots"
EVAL = OUT / "evaluator"
summary = json.loads((OUT / "run_summary.json").read_text())


In [2]:
quality = pd.read_csv(OUT / "data_quality_report.csv")
dictionary = pd.read_csv(OUT / "data_dictionary.csv")
quality


,table,rows,columns,missing_cells,duplicate_rows,data_quality_tier
0,materials,120,10,0,0,CONTROLLED_SYNTHETIC_GROUND_TRUTH
1,finished_goods,24,5,0,0,CONTROLLED_SYNTHETIC_GROUND_TRUTH
2,bom_components,211,9,211,0,CONTROLLED_SYNTHETIC_GROUND_TRUTH
3,production_plan_actuals,1728,8,0,0,CONTROLLED_SYNTHETIC_GROUND_TRUTH
4,material_demand,8640,19,0,0,CONTROLLED_SYNTHETIC_GROUND_TRUTH
5,initial_inventory,120,6,0,0,CONTROLLED_SYNTHETIC_GROUND_TRUTH


In [3]:
materials = pd.read_csv(DATA / "materials.csv")
bom = pd.read_csv(DATA / "bom_components.csv")
production = pd.read_csv(DATA / "production_plan_actuals.csv", parse_dates=["month"])
demand = pd.read_csv(DATA / "material_demand.csv", parse_dates=["month"])
display(materials.head())
display(bom.head())
display(production.head())
display(demand.head())


,material_id,material_code,material_type,description,moq,order_multiple,lead_time_days,unit_cost,service_level,data_quality_tier
0,1,RM-0001,raw_material,Controlled raw material 001,600,200,12,15.82,0.990,CONTROLLED_SYNTHETIC_GROUND_TRUTH
1,2,RM-0002,raw_material,Controlled raw material 002,350,50,41,40.39,0.990,CONTROLLED_SYNTHETIC_GROUND_TRUTH
2,3,RM-0003,raw_material,Controlled raw material 003,150,50,19,11.15,0.990,CONTROLLED_SYNTHETIC_GROUND_TRUTH
3,4,RM-0004,raw_material,Controlled raw material 004,50,10,13,5.91,0.990,CONTROLLED_SYNTHETIC_GROUND_TRUTH
4,5,RM-0005,raw_material,Controlled raw material 005,175,25,11,38.59,0.975,CONTROLLED_SYNTHETIC_GROUND_TRUTH


,bom_version,fg_id,material_id,quantity_per_fg,yield_rate,scrap_rate,effective_from,effective_to,provenance
0,V1_VALIDATED_CONTROL,1,69,0.612122,0.95145,0.04935,2020-01-01,NaN,CONTROLLED_SYNTHETIC_GROUND_TRUTH
1,V1_VALIDATED_CONTROL,1,2,0.377202,0.96132,0.05722,2020-01-01,NaN,CONTROLLED_SYNTHETIC_GROUND_TRUTH
2,V1_VALIDATED_CONTROL,1,87,0.279661,0.97301,0.00738,2020-01-01,NaN,CONTROLLED_SYNTHETIC_GROUND_TRUTH
3,V1_VALIDATED_CONTROL,1,40,0.625181,0.96233,0.04892,2020-01-01,NaN,CONTROLLED_SYNTHETIC_GROUND_TRUTH
4,V1_VALIDATED_CONTROL,1,52,1.108214,0.96571,0.02646,2020-01-01,NaN,CONTROLLED_SYNTHETIC_GROUND_TRUTH


,month,fg_id,planned_fg_units,actual_fg_units,promotion_flag,holiday_flag,shock_type,structural_shift_active
0,2020-01-01,1,3382.331,2893.875,False,True,none,False
1,2020-02-01,1,3621.622,3691.788,False,False,none,False
2,2020-03-01,1,3548.518,4289.630,False,False,none,False
3,2020-04-01,1,3303.924,4229.570,False,True,none,False
4,2020-05-01,1,3843.457,3838.658,True,False,disruption,False


,month,material_id,planned_bom_requirement,actual_bom_requirement,promotion_flag,holiday_flag,shock_flag,active_fg_count,material_code,material_type,description,moq,order_multiple,lead_time_days,unit_cost,service_level,data_quality_tier,demand_units,source
0,2020-01-01,1,394.687692,422.329388,False,True,False,1,RM-0001,raw_material,Controlled raw material 001,600,200,12,15.82,0.990,CONTROLLED_SYNTHETIC_GROUND_TRUTH,418.397,CONTROLLED_SYNTHETIC_GROUND_TRUTH
1,2020-01-01,2,1922.559815,1731.814445,False,True,False,2,RM-0002,raw_material,Controlled raw material 002,350,50,41,40.39,0.990,CONTROLLED_SYNTHETIC_GROUND_TRUTH,1758.340,CONTROLLED_SYNTHETIC_GROUND_TRUTH
2,2020-01-01,3,6379.785131,7330.985779,False,True,False,2,RM-0003,raw_material,Controlled raw material 003,150,50,19,11.15,0.990,CONTROLLED_SYNTHETIC_GROUND_TRUTH,7141.786,CONTROLLED_SYNTHETIC_GROUND_TRUTH
3,2020-01-01,4,661.635622,596.017862,False,True,False,1,RM-0004,raw_material,Controlled raw material 004,50,10,13,5.91,0.990,CONTROLLED_SYNTHETIC_GROUND_TRUTH,601.500,CONTROLLED_SYNTHETIC_GROUND_TRUTH
4,2020-01-01,5,8321.723480,8147.500060,False,True,False,2,RM-0005,raw_material,Controlled raw material 005,175,25,11,38.59,0.975,CONTROLLED_SYNTHETIC_GROUND_TRUTH,8099.795,CONTROLLED_SYNTHETIC_GROUND_TRUTH


In [4]:
lineage = {
    "seed": summary["seed"], "tier": summary["data_tier"],
    "materials": summary["materials"], "finished_goods": summary["finished_goods"],
    "bom_rows": summary["bom_rows"], "controlled_bom_coverage_pct": summary["bom_coverage_controlled_pct"],
}
lineage


{'seed': 20260711,
 'tier': 'CONTROLLED_SYNTHETIC_GROUND_TRUTH',
 'materials': 120,
 'finished_goods': 24,
 'bom_rows': 211,
 'controlled_bom_coverage_pct': 100.0}